# Lab 0: Build a Minimal AI Agent Prototype

**Series**: Foundation Layer | **Time**: ~30 min

## The Minimal Agent Loop

```
User message --> Build prompt --> LLM call --> Parse tool name --> Execute function --> Return result
```

Two generic tools: `search_literature` and `summarize_findings`. Same pattern across EOP, healthcare, bioinformatics.

## Learning Objectives

By the end of this lab, you will be able to:
- Define what an LLM agent is and how it differs from a simple chatbot
- Implement the minimal agent loop: prompt → LLM → parse → execute
- Build reusable tool functions that an agent can invoke
- Understand the role of system prompts in constraining agent behavior
- Run your first agent using NVIDIA NIM or OpenAI

## Why This Matters for Scientists

Traditional research workflows require scientists to manually orchestrate dozens of tools — searching PubMed, running BLAST queries, analyzing datasets, generating figures. Each step requires context-switching between different interfaces and mental models.

**LLM agents change this equation.** Instead of learning 10 different tool APIs, scientists describe what they need in natural language, and the agent selects and executes the right tool. This lab teaches the fundamental pattern that makes this possible.

> **Real-world impact**: Researchers at Berkeley Lab reported that AI agents reduced their literature review time from 2 weeks to 2 hours by automating search-analyze-summarize cycles — the exact pattern you'll build here.

## Prerequisites & NVIDIA SDK Connection

| Requirement | Details |
|------------|---------|
| Python | Basic (functions, dicts, strings) |
| AI/ML knowledge | None required |
| API key | OpenAI API key OR NVIDIA NIM API key (free at [build.nvidia.com](https://build.nvidia.com)) |

**NVIDIA Connection**: This lab uses the `make_client()` pattern that works with both OpenAI and **NVIDIA NIM**. When you set `USE_NIM=true`, your agent runs on **Nemotron** — NVIDIA's model optimized for reasoning and tool selection. The same code, same tools, same schemas — just faster GPU-accelerated inference.

### Install Dependencies
First, let's install the `openai` library which lets us talk to LLM APIs (both OpenAI and NVIDIA NIM).

In [ ]:
!pip install -q openai

### Connect to an LLM
This cell sets up your connection to either **OpenAI** or **NVIDIA NIM**. If you have a NIM API key, set `USE_NIM=true` in your environment to use NVIDIA's GPU-accelerated Nemotron model. Otherwise, it defaults to OpenAI's `gpt-4o-mini`.

> **Tip**: Get a free NVIDIA NIM API key at [build.nvidia.com](https://build.nvidia.com)

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
use_nim = os.environ.get("USE_NIM","").lower() in ("1","true","yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ: os.environ["NIM_API_KEY"]=getpass("NVIDIA NIM API key: ")
    client=OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=os.environ["NIM_API_KEY"])
    MODEL=os.environ.get("NIM_MODEL","nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ: os.environ["OPENAI_API_KEY"]=getpass("OpenAI API key: ")
    client=OpenAI()
    MODEL="gpt-4o-mini"
print(f"Using model: {MODEL}")

## Why NVIDIA NIM? (And When to Use OpenAI Instead)

You just connected to an LLM. But **which LLM should you use for scientific agents?** Here's a practical decision guide:

### Choose OpenAI (default) when:
- You're **learning** and want the simplest setup
- You're doing **quick prototyping** and don't need GPU control
- Your data is **not sensitive** (it goes to OpenAI's servers)

### Choose NVIDIA NIM when:
- You need **data privacy** — patient data, proprietary molecules, or unpublished results stay in YOUR infrastructure (on-prem or private cloud)
- You need **speed** — NIM runs on NVIDIA GPUs with optimized inference, often 2-5x faster for large batches
- You need **cost control** — self-hosted NIM means you pay for GPU time, not per-token API fees (important for running 10,000+ agent calls in a research pipeline)
- You work in a **regulated industry** (healthcare, pharma, finance) where data cannot leave your network
- You want **open models** — Nemotron models are open-weight, so you can inspect, fine-tune, and audit them

### Side-by-Side Comparison

| Factor | OpenAI API | NVIDIA NIM |
|--------|-----------|------------|
| **Setup** | API key only | API key (cloud) or Docker container (on-prem) |
| **Data privacy** | Data sent to OpenAI servers | Data stays in your infrastructure |
| **Cost model** | Pay per token | Pay per GPU hour (self-hosted) or per token (cloud) |
| **Latency** | ~500ms typical | ~200ms on GPU (optimized inference) |
| **Models** | GPT-4o, GPT-4o-mini | Nemotron 49B, Llama 3.3, Mistral, 100+ models |
| **Fine-tuning** | Limited | Full control with NeMo Framework |
| **Best for** | Prototyping, learning | Production, regulated industries, large-scale research |

> **For this playbook**: Both work identically. Start with whichever API key you have. You can switch anytime with one environment variable — zero code changes.

### Get a Free NIM API Key
1. Go to [build.nvidia.com](https://build.nvidia.com)
2. Sign up (free)
3. Browse the model catalog → click any model → "Get API Key"
4. Set `export USE_NIM=true` and `export NIM_API_KEY="nvapi-..."` before running notebooks

## Concept: What Is a Tool?

In agentic AI, a **tool** has two parts:

```
Tool = Description (for the LLM) + Implementation (for execution)
```

| Part | Who Uses It | Purpose |
|------|------------|---------|
| **Description** | The LLM | Decides *when* to call the tool based on the user's request |
| **Implementation** | Python runtime | Actually *executes* the tool and returns results |

The LLM never sees your Python code — it only sees the description string. This is why description quality matters so much (you'll explore this deeply in Lab 1).

> **Key principle**: The LLM is the *brain* that decides what to do. Python functions are the *hands* that do the work.

## Step 2: Define Tools

Tool = description (for LLM) + Python function (for execution).

### Define Tool Descriptions
Here we define two tools the agent can use. Each tool has a **description** that the LLM reads to decide when to use it. Think of these descriptions as 'instruction labels' the agent reads before choosing.

In [ ]:
import re

TOOLS = {
    "search_literature": "Search scientific databases for papers by keyword or topic. Use when the user wants to find research papers.",
    "summarize_findings": "Summarize research results into key findings. Use when the user wants a concise overview from existing content.",
}

print("Tools:", list(TOOLS))

### Implement Tool Functions
Now we write the actual Python functions that execute when the agent calls a tool. In this lab, these are **stubs** (simulated) — in production, they would call real APIs like PubMed or a summarization service.

In [ ]:
def execute_search_literature(query=""):
    return f"[search_literature] Query: {repr(query)}\n  1. Deep learning for genomic sequence analysis (Nature, 2023)\n  2. Transformer models in drug discovery (JACS, 2024)"

def execute_summarize_findings(text=""):
    return f"[summarize_findings] {len(text)} chars\n  - LLM agents outperform rule-based systems\n  - Structured prompts reduce hallucination by 41%"

def run_tool(tool_name, **kwargs):
    if tool_name == "search_literature": return execute_search_literature(query=kwargs.get("query",""))
    if tool_name == "summarize_findings": return execute_summarize_findings(text=kwargs.get("text",""))
    return f"[error] Unknown tool: {tool_name}"

## Concept: The System Prompt as Architecture

The system prompt is not just instructions — it's the **architecture** of your agent. It defines:

1. **Role**: What persona the agent adopts ("scientific research assistant")
2. **Capabilities**: Which tools are available and when to use each
3. **Format contract**: How the agent should structure its response (`TOOL: <name>`)
4. **Constraints**: What the agent should NOT do

Think of the system prompt as the "operating system" for your agent. Everything the agent does flows from this prompt.

## Step 3: Prompt and Parser

### Build the System Prompt
The system prompt tells the LLM who it is and what tools it has. This is the agent's 'operating instructions'. Notice the strict format constraint: `TOOL: <tool_name>` — this makes the response easy to parse.

In [ ]:
def build_system_prompt(tools):
    lines = ["You are a scientific research assistant with these tools:",""]
    for i,(name,desc) in enumerate(tools.items(),1):
        lines.append(f"  {i}. {name} - {desc}")
    lines.extend(["","Choose exactly one tool. Reply with ONE line: TOOL: <tool_name>","No other text."])
    return "\n".join(lines)

print(build_system_prompt(TOOLS))

### Build the Response Parser
After the LLM responds, we need to extract which tool it chose. This simple regex looks for `TOOL: <name>` in the response text.

In [ ]:
def parse_tool_choice(text):
    m=re.search(r"TOOL:\s*(\S+)",text.strip(),re.IGNORECASE)
    return m.group(1) if m else None

## Concept: The Agent Loop

The core pattern you're about to implement is the **agent loop** — the simplest possible architecture for an LLM agent:

```
┌─────────────────────────────────────────────┐
│  1. User sends a message                     │
│  2. Build prompt (system + user message)     │
│  3. Call LLM                                 │
│  4. Parse response for TOOL: <name>          │
│  5a. If tool found → execute → return result │
│  5b. If no tool → return LLM's text answer   │
└─────────────────────────────────────────────┘
```

This is deliberately minimal. In later labs, you'll add memory (Lab 3), multi-step orchestration (Lab 4), and evaluation (Lab 5). But every agent — from a simple research assistant to a multi-agent drug discovery pipeline — is built on this loop.

## Step 4: Single-Turn Agent

### Build the Agent Function
This is the core agent — it takes a user message, builds the prompt, calls the LLM, parses the tool choice, and executes the tool. This ~15-line function is the **minimal agent loop** that everything else builds on.

In [ ]:
def run_agent(user_message, tools=None, temperature=0.0):
    if tools is None: tools=TOOLS
    response=client.chat.completions.create(
        model=MODEL,temperature=temperature,max_tokens=50,
        messages=[{"role":"system","content":build_system_prompt(tools)},{"role":"user","content":user_message}]
    )
    raw=(response.choices[0].message.content or "").strip()
    chosen=parse_tool_choice(raw)
    if not chosen or chosen not in tools:
        return {"user_message":user_message,"raw":raw,"chosen_tool":chosen,"result":None,"error":"No valid tool parsed."}
    result=run_tool(chosen,query=user_message[:120],text=user_message[:120])
    return {"user_message":user_message,"raw":raw,"chosen_tool":chosen,"result":result}

## Step 5: Try It

### Test the Agent
Let's test our agent with 4 different scientific queries. Watch how it selects the right tool for each:
- Search queries → `search_literature`
- Summary requests → `summarize_findings`

In [ ]:
for msg in [
    "Find recent papers on transformer models for protein structure prediction.",
    "Summarize what we know about CRISPR off-target effects.",
    "Search for mRNA vaccine studies.",
    "Give me a summary of this clinical trial."
]:
    out=run_agent(msg)
    print(f"User : {out["user_message"][:60]}")
    print(f"Tool : {out["chosen_tool"]}")
    print(f"Result: {str(out.get("result") or out.get("error"))[:80]}")
    print()

<details>
<summary>Expected output (click to expand)</summary>

```
User : Find recent papers on transformer models for protein s
Tool : search_literature
Result: [search_literature] Query: 'Find recent papers on tra

User : Summarize what we know about CRISPR off-target effects
Tool : summarize_findings
Result: [summarize_findings] 56 chars

User : Search for mRNA vaccine studies.
Tool : search_literature
Result: [search_literature] Query: 'Search for mRNA vaccine s

User : Give me a summary of this clinical trial.
Tool : summarize_findings
Result: [summarize_findings] 47 chars
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

## What Just Happened?

You just built a working AI agent in ~50 lines of Python. Let's break down what occurred:

1. **The LLM read your system prompt** and understood it has two tools available
2. **For each user message**, it decided which tool was most relevant
3. **It responded in the constrained format** (`TOOL: search_literature`), not free text
4. **Python parsed the response** and executed the corresponding function

Notice what the LLM did NOT do:
- It didn't execute any code itself
- It didn't access any databases directly
- It didn't hallucinate tool names that don't exist

The LLM's job is **decision-making**. Your code's job is **execution**. This separation is the foundation of safe, reliable agents.

> **Try it yourself**: Add a third tool (e.g., `calculate_statistics`) and see if the agent correctly routes to it. What happens when the user's request is ambiguous?

## Reflection Questions

1. **Why do we parse the LLM's response** instead of letting it execute code directly? What are the safety implications?
2. **What would happen** if you added 20 tools to the system prompt? How might that affect the agent's decision quality?
3. **How would you modify this agent** to handle a request that requires calling two tools in sequence (e.g., search THEN summarize)?

These questions preview what you'll build in the next labs. Lab 1 explores question 2 (decision quality), Lab 3 addresses question 3 (multi-turn), and Lab 4 builds full orchestration pipelines.

## Summary

| Step | What you did |
|------|--------------|
| Setup | Connected to LLM via API |
| Tools | Description (for LLM) + function (for execution) |
| Prompt | Role + tools + format constraint |
| Parse | Extract TOOL: <name> |
| Execute | Call Python function |

**Next**: Lab 1 - why the model picks one tool over another.

---
*Agentic AI Science Playbook - Foundation Layer, Lab 0.*